<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l3.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L3 · Drawdown Monte Carlo
Equity y drawdown reconstruidos desde 50 retornos + Monte Carlo de 2.000 caminos reordenados.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/riesgo/data/c4_l3.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c4_l3.csv'), Path('data/c4_l3.csv'), Path('c4_l3.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)

In [ ]:
df['equity_calc'] = 10000 * (1 + df['retorno']).cumprod()
df['peak_calc'] = df['equity_calc'].cummax()
df['dd_calc'] = (df['equity_calc'] - df['peak_calc']) / df['peak_calc'] * 100
print(df[['dia','retorno','equity','equity_calc','drawdown_pct','dd_calc']].head(8).to_string(index=False))
worst = df.loc[df['dd_calc'].idxmin()]
print(f"max DD={worst['dd_calc']:.3f}% en dia {int(worst['dia'])}  equity={worst['equity_calc']:.2f}")

In [ ]:
rng = np.random.default_rng(42)
rets = df['retorno'].to_numpy()
sims = rng.choice(rets, size=(2000, len(rets)))
eq = 10000 * np.cumprod(1 + sims, axis=1)
peak = np.maximum.accumulate(eq, axis=1)
mdd = ((eq - peak) / peak).min(axis=1) * 100
print(f"P(MDD < -5%)={(mdd < -5).mean():.1%}  mediana MDD={np.median(mdd):.2f}%  peor sim={mdd.min():.2f}%")

In [ ]:
# Chequeo automático
assert ((df['equity'] - df['equity_calc']).abs().max() < 5.0), 'equity no replica (CSV redondea a 2 decimales por fila)'
assert abs(df['dd_calc'].min() - df['drawdown_pct'].min()) < 1e-3, 'max DD no coincide'
assert -10 < df['drawdown_pct'].min() < 0
print('OK: drawdown y Monte Carlo verificados')